# **Start Section:**

In [1]:
# FIX K1: same notebook runs on Google Colab (Drive) or Kaggle (internet on).
# On Kaggle the working dir is /kaggle/working, so the original "./drive/MyDrive/..." paths
# below are reused unchanged; the dataset is copied there as train.csv / test.csv.
import os, glob, shutil
IN_KAGGLE = os.path.exists("/kaggle/working")
if IN_KAGGLE:
    _data_dir = "./drive/MyDrive/Regression/noise_uncertainty_analysis/data"
    os.makedirs(_data_dir, exist_ok=True)
    shutil.copy(glob.glob("/kaggle/input/**/d06_a.csv", recursive=True)[0], f"{_data_dir}/train.csv")  # noise train
    shutil.copy(glob.glob("/kaggle/input/**/d06_b.csv", recursive=True)[0], f"{_data_dir}/test.csv")   # noise test
    for _link in ["/content", "/kaggle/content"]:
        if not os.path.exists(_link):
            os.symlink("/kaggle/working", _link)
    print("Kaggle: noise data copied to", _data_dir)
else:
    from google.colab import drive
    drive.mount('/content/drive')


Kaggle: noise data copied to ./drive/MyDrive/Regression/noise_uncertainty_analysis/data


In [2]:
# FIX K1: the treeffuser fork pins numpy<2 / scikit-learn==1.5.0 / lightgbm==4.3.0, which cannot be
# installed on Kaggle's Python 3.13. On Kaggle install the same code without those pins.
if IN_KAGGLE:
    !pip install -q --no-deps git+https://github.com/DaneshSelwal/treeffuser.git
    !pip install -q jaxtyping einops ml-collections
else:
    !pip install git+https://github.com/DaneshSelwal/treeffuser.git


  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
treeffuser 0.1.3 requires jaxtyping<0.3.0,>=0.2.19, but you have jaxtyping 0.3.11 which is incompatible.
treeffuser 0.1.3 requires lightgbm==4.3.0, but you have lightgbm 4.6.0 which is incompatible.
treeffuser 0.1.3 requires ml-collections<0.2.0,>=0.1.1, but you have ml-collections 1.1.0 which is incompatible.
treeffuser 0.1.3 requires numpy<2.0,>=1.24, but you have numpy 2.1.3 which is incompatible.
treeffuser 0.1.3 requires scikit-learn==1.5.0, but you have scikit-learn 1.6.1 which is incompatible.


In [3]:
# FIX K1: original Colab install block kept as-is (runs only on Colab).
# On Kaggle: install only the packages the Kaggle image lacks, pinned to the Colab versions,
# and do NOT restart the kernel with os._exit(0).
if IN_KAGGLE:
    !pip install -q optuna==4.6.0 optunahub==0.4.0 cmaes==0.12.0 xgboost==3.1.2 catboost==1.2.8 gpboost==1.6.1 ngboost==0.5.8 pgbm==2.2.0 pytorch-tabnet2==4.5.3 interpret==0.7.4 lime==0.2.0.1 mapie==0.6.0 puncc==0.8.0 properscoring==0.1 XlsxWriter==3.2.9
else:
    import os
    os.environ["PIP_CONSTRAINT"] = "/tmp/numpy_constraint.txt"
    !echo "numpy==1.26.4" > /tmp/numpy_constraint.txt

    !pip uninstall -y torch torchaudio torchvision \
        torchao torchcodec torchdata torchtune torchsummary -q 2>/dev/null

    !pip uninstall -y tensorflow tensorflow-text tensorflow-hub tf-keras \
        tensorflow_decision_forests tensorflow-probability \
        tensorflow-Datasets tensorflow-metadata -q 2>/dev/null

    !pip uninstall -y numpy scikit-learn shap xgboost lightgbm dask \
        seaborn plotly openpyxl Cython catboost interpret lime -q 2>/dev/null

    !pip install numpy==1.26.4 -q
    !pip install scikit-learn==1.6.1 -q
    !pip install torch==2.9.0 -q

    !pip install lightgbm==4.6.0 -q
    !pip install xgboost==3.1.2 -q
    !pip install catboost==1.2.8 -q
    !pip install gpboost==1.6.1 -q
    !pip install ngboost==0.5.8 -q
    !pip install pgbm==2.2.0 -q
    !pip install pytorch-tabnet2==4.5.3 -q

    !pip install bayesian-optimization==3.2.0 -q
    !pip install optuna==4.6.0 -q
    !pip install optunahub==0.4.0 -q
    !pip install cmaes==0.12.0 -q

    !pip install shap==0.44.0 -q
    !pip install lime==0.2.0.1 -q
    !pip install interpret==0.7.4 -q

    !pip install mapie==0.6.0 -q
    !pip install puncc==0.8.0 -q
    !pip install skorch==1.3.1 -q
    !pip install properscoring==0.1 -q

    !pip install dask[dataframe]==2025.12.0 -q
    !pip install cython==3.0.12 -q
    !pip install seaborn==0.13.2 -q
    !pip install plotly==5.24.1 -q
    !pip install kaleido==1.2.0 -q
    !pip install openpyxl==3.1.5 -q
    !pip install XlsxWriter==3.2.9 -q
    !pip install cp==2020.12.3 -q

    !pip install numpy==1.26.4 --force-reinstall --no-deps -q

    os._exit(0)


  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


# **Imports**

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ngboost
from scipy.stats import randint
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.svm import SVR
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from gpboost import GPBoostRegressor
from ngboost import NGBRegressor
import optuna
import optunahub
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingGridSearchCV
from interpret import show
from interpret.blackbox import LimeTabular, ShapKernel
from optuna.samplers import RandomSampler
import random
import time
from ngboost.distns import Normal
from ngboost.scores import LogScore
from scipy.stats import norm
from interpret import set_visualize_provider
from interpret.provider import InlineProvider
from interpret.glassbox import ExplainableBoostingRegressor
from interpret import show
import plotly.express as px
from io import BytesIO
from openpyxl import Workbook, load_workbook
import os
import properscoring as ps
import io
from openpyxl.drawing.image import Image as openpyxlImage
import warnings
import xlsxwriter
from pgbm.sklearn import HistGradientBoostingRegressor
import torch
from pgbm.torch import PGBM
import plotly.graph_objects as go
warnings.filterwarnings('ignore')
import pickle
import json
from mapie.subsample import Subsample
from mapie.regression import MapieRegressor
from deel.puncc.metrics import regression_sharpness, regression_mean_coverage
from deel.puncc.api.prediction import BasePredictor, DualPredictor
from deel.puncc.regression import SplitCP, CVPlus, CQR
from deel.puncc.plotting import plot_prediction_intervals
from sklearn.model_selection import train_test_split
from typing_extensions import TypedDict
from typing import Union
from mapie.metrics import regression_coverage_score
from sklearn.model_selection import KFold
from PIL import Image as PImage
from pathlib import Path
from scipy.stats import norm, kstest
from pytorch_tabnet import TabNetRegressor
from treeffuser import Treeffuser, Samples
from treeffuser._score_models import TreeBasedScoreModel
import torch
import torch.nn as nn
import torch.optim as optim

[1/2] c++ -MMD -MF splitgain_cpu.o.d -DTORCH_EXTENSION_NAME=split_decision -DTORCH_API_INCLUDE_EXTENSION_H -isystem /usr/local/lib/python3.13/dist-packages/torch/include -isystem /usr/local/lib/python3.13/dist-packages/torch/include/torch/csrc/api/include -isystem /usr/include/python3.13 -fPIC -std=c++17 -c /usr/local/lib/python3.13/dist-packages/pgbm/torch/splitgain_cpu.cpp -o splitgain_cpu.o 


[2/2] c++ splitgain_cpu.o -shared -L/usr/local/lib/python3.13/dist-packages/torch/lib -lc10 -ltorch_cpu -ltorch -ltorch_python -o split_decision.so


I0000 00:00:1795378996.538485      34 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


I0000 00:00:1795379013.696282      34 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [5]:
# Go to find & replace button and replace (noise_uncertainty_analysis) with your folder name. Rename your train and test Dataset as train.csv and test.csv.
# Modify the names of the feature in the below cell.
# Replace (str) with actual data label name.

In [6]:
feature_names = ['A', 'B', 'C', 'D', 'O']
TARGET_LABEL = 'Noise level (Leq)'  # FIX B12: target name used in plot labels (was 'str', the concrete dataset's target, on every dataset)

In [7]:
train_data_path = "./drive/MyDrive/Regression/noise_uncertainty_analysis/data/train.csv"
test_data_path = "./drive/MyDrive/Regression/noise_uncertainty_analysis/data/test.csv"
train_data = pd.read_csv(train_data_path)
test_data = pd.read_csv(test_data_path)
print("Training data loaded successfully.")
print("Test data loaded successfully.")

Training data loaded successfully.
Test data loaded successfully.


In [8]:
print("\nShape of training data:", train_data.shape)
print("First 5 rows of training data:\n", train_data.head(5))
print("\nShape of test data:", test_data.shape)
print("First 5 rows of test data:\n", test_data.head(5))


Shape of training data: (168, 6)
First 5 rows of training data:
       A    B    C    D   O    Leq
0   894  354  228   30  42  70.06
1  1068  312  234   24  24  69.14
2  1506  894  198  222  18  73.02
3  1188  498  252  354  18  75.71
4  1068  438  210  366  30  70.78

Shape of test data: (60, 6)
First 5 rows of test data:
       A    B    C    D   O    Leq
0  1032  282   90  108   6  70.07
1  1188  498  252  408  18  72.90
2   570   90   60   84  24  68.66
3   450  156  150   72   0  50.24
4   240   66   18   24   0  40.17


In [9]:
X_train = train_data.iloc[:, :-1]
y_train = train_data.iloc[:, -1]
X_test = test_data.iloc[:, :-1]
y_test = test_data.iloc[:, -1]
# FIX D4 / D5 (approved 2026-11-22): settings that follow the test-set size
N_BINS = max(2, min(10, len(y_test) // 10))  # D4: bins for binned metrics (was 10 on every dataset; pile: 2-3 points per bin)
COP_T_BURNIN = min(50, len(y_test) // 2)  # D5: COP burn-in (was 50, so with < 50 test points COP's correction never started)
x_test= X_test
print("\nShape of X_train:", X_train.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of y_test:", y_test.shape)


Shape of X_train: (168, 5)
Shape of y_train: (168,)
Shape of X_test: (60, 5)
Shape of y_test: (60,)


In [10]:
# Apply z-score normalization
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Print the first five rows of the normalized data
print("\nFirst five rows of normalized X_train:")
print(X_train[:5])

print("\nFirst five rows of normalized X_test:")
print(X_test[:5])


First five rows of normalized X_train:
[[-0.34508703 -0.52871218  0.0597039  -1.05667951  1.31930842]
 [-0.08377773 -0.63289541  0.09903824 -1.09452265  0.25541811]
 [ 0.57400083  0.8107865  -0.13696778  0.15430089 -0.09921199]
 [ 0.09643557 -0.17151253  0.21704125  0.98684991 -0.09921199]
 [-0.08377773 -0.32034572 -0.05829911  1.06253618  0.61004822]]

First five rows of normalized X_test:
[[-0.13784172 -0.707312   -0.84498584 -0.56471872 -0.8084722 ]
 [ 0.09643557 -0.17151253  0.21704125  1.32743815 -0.09921199]
 [-0.83166295 -1.1835782  -1.04165752 -0.71609127  0.25541811]
 [-1.01187626 -1.01986169 -0.45164247 -0.79177755 -1.16310231]
 [-1.32724954 -1.24311147 -1.31699788 -1.09452265 -1.16310231]]


# **Functions:**

In [11]:
# Define the model classes
model_classes = {
    'Random Forest': RandomForestRegressor,
    'Gradient Boosting': GradientBoostingRegressor,
    'XGBoost': XGBRegressor,
    'LightGBM': LGBMRegressor,
    'GPBoost': GPBoostRegressor,
    'CatBoost': CatBoostRegressor,
    'HistGradientBoosting': HistGradientBoostingRegressor,
    'TabNet': TabNetRegressor,
    'NGBoost': NGBRegressor
}

def _ensure_excel_file(path):
    import os
    import pandas as pd
    if not os.path.exists(path):
        parent = os.path.dirname(path)
        if parent:
            os.makedirs(parent, exist_ok=True)
        pd.DataFrame().to_excel(path)
    return path

In [12]:
def get_best_model_params(results, model_name):
    # Map model names to dictionary keys, assuming keys are strings like 'XGBoost' and not objects
    model_keys = {
        'LightGBM': 'LightGBM',
        'XGBoost': 'XGBoost',
        'GPBoost': 'GPBoost',
        'NGBoost': 'NGBoost',
        'GradientBoosting': 'Gradient Boosting',
        'HGBR' : 'HistGradientBoosting',
        'CatBoost' : 'CatBoost',
        'TabNet' : 'TabNet',
        'PGBM' : 'PGBM'
    }

    # Ensure the requested model name is valid
    if model_name not in model_keys:
        raise ValueError(f"Model name '{model_name}' is not recognized. Available models are: {list(model_keys.keys())}")

    # Filter out entries for the specified model
    model_entries = {key: value for key, value in results.items() if key[0] == model_keys[model_name]}

    # Find the entry with the best (lowest) 'best_score'
    best_entry_key, best_entry_value = min(model_entries.items(), key=lambda item: item[1]['best_score'])

    # Return the best hyperparameters
    return best_entry_value['best_params']

In [13]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def generate_and_save_plots(predictions_df, y_test, excel_path):
    # 1. Predicted vs. Actual Plot
    plt.figure(figsize=(12, 8))
    sns.scatterplot(x='Actual', y='Mean', data=predictions_df)
    plt.plot([predictions_df['Actual'].min(), predictions_df['Actual'].max()],
             [predictions_df['Actual'].min(), predictions_df['Actual'].max()],
             color='red', linestyle='--')
    plt.title(f'Predicted {TARGET_LABEL} vs. Actual {TARGET_LABEL}')
    plt.xlabel(f'Actual {TARGET_LABEL}')
    plt.ylabel(f'Predicted Mean {TARGET_LABEL}')
    plt.tight_layout()
    predicted_vs_actual_path = 'predicted_vs_actual.png'
    _ensure_parent_dir(predicted_vs_actual_path)
    plt.savefig(_ensure_parent_dir(predicted_vs_actual_path))
    plt.close()

    # 2. Uncertainty Bands
    plt.figure(figsize=(12, 8))
    plt.errorbar(x=range(len(predictions_df)), y=predictions_df['Mean'],
                 yerr=predictions_df['StdDev'], fmt='o', markersize=4, ecolor='red', elinewidth=1, capsize=5, alpha=0.7)
    plt.plot(range(len(predictions_df)), predictions_df['Mean'], color='black', linewidth=0.5, alpha=0.7)
    plt.title('Predicted Mean with Uncertainty Bands', fontsize=16)
    plt.xlabel('Sample Number', fontsize=14)
    plt.ylabel(f'Predicted Mean {TARGET_LABEL}')
    plt.tight_layout()
    uncertainty_bands_path = 'uncertainty_bands.png'
    _ensure_parent_dir(uncertainty_bands_path)
    plt.savefig(_ensure_parent_dir(uncertainty_bands_path))
    plt.close()

    # 3. Histogram of Standard Deviations
    plt.figure(figsize=(12, 8))
    sns.histplot(predictions_df['StdDev'], bins=20, kde=True)
    plt.title('Distribution of Predicted Standard Deviations')
    plt.xlabel('Standard Deviation')
    plt.ylabel('Frequency')
    plt.tight_layout()
    histogram_stddev_path = 'histogram_stddev.png'
    _ensure_parent_dir(histogram_stddev_path)
    plt.savefig(_ensure_parent_dir(histogram_stddev_path))
    plt.close()

    # 4. Residual Plot
    residuals = predictions_df['Actual'] - predictions_df['Mean']
    plt.figure(figsize=(12, 8))
    sns.scatterplot(x=predictions_df['Mean'], y=residuals)
    plt.axhline(0, color='red', linestyle='--')
    plt.title('Residual Plot')
    plt.xlabel(f'Predicted Mean {TARGET_LABEL}')
    plt.ylabel('Residuals')
    plt.tight_layout()
    residual_plot_path = 'residual_plot.png'
    _ensure_parent_dir(residual_plot_path)
    plt.savefig(_ensure_parent_dir(residual_plot_path))
    plt.close()

    # Plot the standard deviation of the predictions
    plt.figure(figsize=(12, 6))
    plt.plot(
        predictions_df['StdDev'],
        label='Standard Deviation',
        color='green',
        marker='o',
        linestyle='dashed'
    )
    plt.xlabel('Sample Number')
    plt.ylabel('Standard Deviation')
    plt.title('Standard Deviation of Predictions')
    plt.legend(loc='upper left', fontsize='small', bbox_to_anchor=(1.05, 1), borderaxespad=0.)
    plt.tight_layout()
    stddev_plot_path = 'stddev_plot.png'
    _ensure_parent_dir(stddev_plot_path)
    plt.savefig(_ensure_parent_dir(stddev_plot_path))
    plt.close()

    # Plot the actual vs predicted mean with confidence intervals
    plt.figure(figsize=(12, 6))
    plt.plot(
        y_test,
        label=f'Actual {TARGET_LABEL}',
        color='black',
        marker='o',
        linestyle='dashed',
        linewidth=1,
        markersize=4
    )
    plt.plot(
        predictions_df['Mean'],
        label=f'Predicted Mean {TARGET_LABEL}',
        color='red',
        marker='x',
        linestyle='dashed',
        linewidth=1,
        markersize=4
    )
    confidence_interval = 1.96 * predictions_df['StdDev']
    plt.fill_between(
        np.arange(len(predictions_df)),
        predictions_df['Mean'] - confidence_interval,
        predictions_df['Mean'] + confidence_interval,
        color='blue',
        alpha=0.2,
        label='95% Confidence Interval'
    )
    plt.xlabel('Sample Number')
    plt.ylabel(f' {TARGET_LABEL}')
    plt.title('Predictions with Confidence Intervals')
    plt.legend(loc='upper left', fontsize='small', bbox_to_anchor=(1.05, 1), borderaxespad=0.)
    plt.tight_layout()
    prediction_plot_path = 'prediction_plot.png'
    _ensure_parent_dir(prediction_plot_path)
    plt.savefig(_ensure_parent_dir(prediction_plot_path))
    plt.close()

    # Create a Pandas Excel writer using XlsxWriter as the engine.
    writer = pd.ExcelWriter(_ensure_parent_dir(excel_path), engine='xlsxwriter')

    # Write the dataframe to a worksheet.
    writer
    predictions_df.to_excel(writer, sheet_name='Predictions', index=False)

    # Access the XlsxWriter workbook and worksheet objects from the writer object.
    workbook = writer.book

    # Insert the plots into new worksheets.
    worksheet_predicted_vs_actual = workbook.add_worksheet('Predicted vs Actual')
    worksheet_predicted_vs_actual.insert_image('A1', predicted_vs_actual_path)

    worksheet_uncertainty_bands = workbook.add_worksheet('Uncertainty Bands')
    worksheet_uncertainty_bands.insert_image('A1', uncertainty_bands_path)

    worksheet_histogram_stddev = workbook.add_worksheet('Histogram StdDev')
    worksheet_histogram_stddev.insert_image('A1', histogram_stddev_path)

    worksheet_residual_plot = workbook.add_worksheet('Residual Plot')
    worksheet_residual_plot.insert_image('A1', residual_plot_path)

    worksheet_stddev = workbook.add_worksheet('StdDev Plot')
    worksheet_stddev.insert_image('A1', stddev_plot_path)

    worksheet_prediction = workbook.add_worksheet('Predictions Plot')
    worksheet_prediction.insert_image('A1', prediction_plot_path)

    # Close the Pandas Excel writer and output the Excel file.
    writer.close()
    print(f"Excel file saved at {excel_path}")

    # Optionally, remove the image files now that they're embedded in the Excel file
    if os.path.exists(str(predicted_vs_actual_path)): os.remove(str(predicted_vs_actual_path))
    if os.path.exists(str(uncertainty_bands_path)): os.remove(str(uncertainty_bands_path))
    if os.path.exists(str(histogram_stddev_path)): os.remove(str(histogram_stddev_path))
    if os.path.exists(str(residual_plot_path)): os.remove(str(residual_plot_path))
    if os.path.exists(str(stddev_plot_path)): os.remove(str(stddev_plot_path))
    if os.path.exists(str(prediction_plot_path)): os.remove(str(prediction_plot_path))

In [14]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def generate_and_save_crps_decomposition(
    predictions_df,
    y_true,
    excel_path,
    model_name="Model",
    n_bins=N_BINS
):

    # -------------------------------
    # Input handling
    # -------------------------------
    mu_pred = predictions_df['Mean'].values
    sigma_pred = predictions_df['StdDev'].values

    if 'Actual' in predictions_df.columns:
        y_true = predictions_df['Actual'].values
    else:
        y_true = np.asarray(y_true).ravel()

    y_true = np.asarray(y_true).ravel()

    # -------------------------------
    # Total CRPS
    # -------------------------------
    crps_values = ps.crps_gaussian(
        y_true, mu=mu_pred, sig=sigma_pred
    )
    total_crps = np.mean(crps_values)

    # -------------------------------
    # FIX P1: Reliability / Resolution / Uncertainty from the Hersbach (2000) decomposition,
    # "Decomposition of the CRPS for ensemble prediction systems", Weather and Forecasting 15.
    # Was: Reliability = mean of the model's own CRPS over 10 groups (= total CRPS), so
    # Resolution always came out = Uncertainty and the split carried no information.
    # Each predicted Gaussian is represented by M members at its quantile levels (k-0.5)/M.
    #   CRPS_ens = Reliability + CRPS_pot,  CRPS_pot = Uncertainty - Resolution
    # -------------------------------
    M = 100
    sig_safe = np.clip(sigma_pred, 1e-12, None)
    levels = (np.arange(1, M + 1) - 0.5) / M
    ens = np.sort(norm.ppf(levels[None, :], loc=mu_pred[:, None], scale=sig_safe[:, None]), axis=1)  # (n, M)
    yy = y_true[:, None]
    n_obs = len(y_true)

    # alpha_i, beta_i for the inner bins [x_i, x_{i+1}], i = 1..M-1 (Hersbach eq. 26)
    lo, hi = ens[:, :-1], ens[:, 1:]
    alpha_in = np.where(yy > hi, hi - lo, np.where(yy > lo, yy - lo, 0.0))
    beta_in = np.where(yy < lo, hi - lo, np.where(yy < hi, hi - yy, 0.0))
    alpha_bar = np.zeros(M + 1)
    beta_bar = np.zeros(M + 1)
    alpha_bar[1:M] = alpha_in.mean(axis=0)
    beta_bar[1:M] = beta_in.mean(axis=0)
    # outliers (Hersbach eq. 27): below the lowest / above the highest member
    beta_bar[0] = np.mean(np.where(y_true < ens[:, 0], ens[:, 0] - y_true, 0.0))
    alpha_bar[M] = np.mean(np.where(y_true > ens[:, -1], y_true - ens[:, -1], 0.0))

    p = np.arange(M + 1) / M
    g = alpha_bar + beta_bar
    o = np.divide(beta_bar, g, out=np.zeros_like(g), where=g > 0)
    # outlier bins: o_0 = freq(y < x_1), o_M = 1 - freq(y > x_M) (Hersbach eq. 33)
    o[0] = np.mean(y_true < ens[:, 0])
    o[M] = 1.0 - np.mean(y_true > ens[:, -1])
    g[0] = beta_bar[0] / o[0] if o[0] > 0 else 0.0
    g[M] = alpha_bar[M] / (1.0 - o[M]) if o[M] < 1 else 0.0

    reliability = float(np.sum(g * (o - p) ** 2))
    crps_pot = float(np.sum(g * o * (1.0 - o)))
    # Uncertainty = CRPS of the sample climatology of the observations
    uncertainty = float(np.abs(y_true[:, None] - y_true[None, :]).sum() / (2.0 * n_obs ** 2))
    resolution = uncertainty - crps_pot

    # -------------------------------
    # Per-bin table (unchanged): test rows sorted by predicted mean, split into n_bins groups
    # -------------------------------
    sort_idx = np.argsort(mu_pred)
    y_sorted = y_true[sort_idx]
    mu_sorted = mu_pred[sort_idx]
    sigma_sorted = sigma_pred[sort_idx]

    bins = np.array_split(np.arange(len(y_true)), n_bins)

    bin_results = []

    for i, b in enumerate(bins):
        bin_y = y_sorted[b]
        bin_mu = mu_sorted[b]
        bin_sigma = sigma_sorted[b]

        bin_crps = np.mean(
            ps.crps_gaussian(bin_y, mu=bin_mu, sig=bin_sigma)
        )

        bin_results.append({
            "Bin": i + 1,
            "Bin_Size": len(b),
            "Mean_Pred_Mu": np.mean(bin_mu),
            "Mean_Pred_Sigma": np.mean(bin_sigma),
            "Bin_CRPS": bin_crps
        })

    # -------------------------------
    # Summary table
    # -------------------------------
    summary_df = pd.DataFrame([{
        "Model": model_name,
        "Total_CRPS": total_crps,
        "Reliability": reliability,
        "Resolution": resolution,
        "Uncertainty": uncertainty,
        "Num_Bins": n_bins
    }])

    bin_df = pd.DataFrame(bin_results)

    # -------------------------------
    # Save to Excel
    # -------------------------------
    excel_path = Path(excel_path)
    excel_path.parent.mkdir(parents=True, exist_ok=True)

    sheet_summary = f"CRPS_Summary_{model_name}"
    sheet_bins = f"CRPS_Bins_{model_name}"

    with pd.ExcelWriter(_ensure_parent_dir(_ensure_excel_file(
        excel_path)),
        engine="openpyxl",
        mode="a" if excel_path.exists() else "w",
        if_sheet_exists="replace"
    ) as writer:
        writer
        summary_df.to_excel(writer, sheet_name=sheet_summary, index=False)
        writer
        bin_df.to_excel(writer, sheet_name=sheet_bins, index=False)

    print(f"✅ CRPS decomposition saved to:\n{excel_path}")
    print(f"   Sheets: {sheet_summary}, {sheet_bins}")

    return summary_df

In [15]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def generate_and_save_pit_results(
    predictions_df,
    y_true,
    excel_path,
    model_name="Model",
    n_bins=N_BINS
):
    """
    Generate PIT values, histogram, calibration diagnostics,
    and save everything to Excel with the PIT plot EMBEDDED.
    """

    # -------------------------------
    # Input handling
    # -------------------------------
    mu_pred = predictions_df['Mean'].values
    sigma_pred = predictions_df['StdDev'].values

    if 'Actual' in predictions_df.columns:
        y_true = predictions_df['Actual'].values
    else:
        y_true = np.asarray(y_true).ravel()

    y_true = np.asarray(y_true).ravel()

    sigma_pred = np.clip(sigma_pred, 1e-6, None)

    # -------------------------------
    # Compute PIT values
    # -------------------------------
    pit_values = norm.cdf(y_true, loc=mu_pred, scale=sigma_pred)

    pit_df = pd.DataFrame({"PIT_Value": pit_values})

    # -------------------------------
    # Histogram data
    # -------------------------------
    hist_density, bin_edges = np.histogram(
        pit_values, bins=n_bins, range=(0, 1), density=True
    )
    bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

    hist_df = pd.DataFrame({
        "Bin_Center": bin_centers,
        "Density": hist_density
    })

    # -------------------------------
    # Calibration diagnostics
    # -------------------------------
    pit_mean = pit_values.mean()
    pit_var = pit_values.var()
    ks_stat, ks_pvalue = kstest(pit_values, 'uniform')

    if ks_pvalue > 0.05:
        diagnosis = "Well-calibrated (Uniform PIT)"
    elif pit_var < 1/12:
        diagnosis = "Over-dispersed (Under-confident)"
    elif pit_var > 1/12:
        diagnosis = "Under-dispersed (Over-confident)"
    else:
        diagnosis = "Biased or miscalibrated"

    summary_df = pd.DataFrame([{
        "Model": model_name,
        "Mean_PIT": pit_mean,
        "Variance_PIT": pit_var,
        "Ideal_Variance_(1/12)": 1/12,
        "KS_Statistic": ks_stat,
        "KS_pvalue": ks_pvalue,
        "Calibration_Diagnosis": diagnosis
    }])

    # -------------------------------
    # Create PIT plot in memory
    # -------------------------------
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.hist(
        pit_values,
        bins=n_bins,
        density=True,
        color="skyblue",
        edgecolor="black",
        alpha=0.75
    )
    ax.axhline(1, color="red", linestyle="--", label="Perfect Calibration")
    ax.set_xlabel("PIT Value")
    ax.set_ylabel("Density")
    ax.set_title(f"PIT Histogram – {model_name}")
    ax.legend()
    fig.tight_layout()

    img_buffer = BytesIO()
    fig.savefig(img_buffer, format="png", dpi=300)
    plt.close(fig)
    img_buffer.seek(0)

    # -------------------------------
    # Save to Excel
    # -------------------------------
    excel_path = Path(excel_path)
    excel_path.parent.mkdir(parents=True, exist_ok=True)

    sheet_values = f"PIT_Values_{model_name}"
    sheet_hist = f"PIT_Histogram_{model_name}"
    sheet_summary = f"PIT_Summary_{model_name}"
    sheet_plot = f"PIT_Plot_{model_name}"

    with pd.ExcelWriter(_ensure_parent_dir(_ensure_excel_file(
        excel_path)),
        engine="openpyxl",
        mode="a" if excel_path.exists() else "w",
        if_sheet_exists="replace"
    ) as writer:

        writer
        pit_df.to_excel(writer, sheet_name=sheet_values, index=False)
        writer
        hist_df.to_excel(writer, sheet_name=sheet_hist, index=False)
        writer
        summary_df.to_excel(writer, sheet_name=sheet_summary, index=False)

        # Create empty sheet for plot
        workbook = writer.book
        worksheet = workbook.create_sheet(sheet_plot)

        img = openpyxlImage(img_buffer)
        img.width = img.width * 0.4
        img.height = img.height * 0.4

        img.anchor = "B2"
        worksheet.add_image(img)

    print(f"✅ PIT values, histogram, summary, and EMBEDDED plot saved to:")
    print(f"   {excel_path}")

    return pit_df

In [16]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def generate_and_save_calibration_curve(
    predictions_df,
    y_true,
    excel_path,
    model_name="Model",
    n_quantiles=10
):
    """
    Generate quantile calibration curve (reliability diagram),
    compute calibration errors, and save everything to Excel
    with the plot EMBEDDED.
    """

    # -------------------------------
    # Input handling
    # -------------------------------
    mu_pred = predictions_df['Mean'].values
    sigma_pred = predictions_df['StdDev'].values

    if 'Actual' in predictions_df.columns:
        y_true = predictions_df['Actual'].values
    else:
        y_true = np.asarray(y_true).ravel()

    y_true = np.asarray(y_true).ravel()
    sigma_pred = np.clip(sigma_pred, 1e-6, None)

    # -------------------------------
    # Quantiles
    # -------------------------------
    quantiles = np.linspace(0, 1, n_quantiles + 1)

    observed_freq = []
    predicted_q = []

    for q in quantiles:
        # Predicted value at quantile q
        thresh = norm.ppf(q, loc=mu_pred, scale=sigma_pred)

        # Observed frequency
        obs = np.mean(y_true <= thresh)

        predicted_q.append(q)
        observed_freq.append(obs)

    predicted_q = np.array(predicted_q)
    observed_freq = np.array(observed_freq)

    # -------------------------------
    # Calibration metrics
    # -------------------------------
    mce = np.mean(np.abs(observed_freq - predicted_q))
    rmsce = np.sqrt(np.mean((observed_freq - predicted_q) ** 2))

    calib_df = pd.DataFrame({
        "Predicted_Quantile": predicted_q,
        "Observed_Frequency": observed_freq,
        "Absolute_Error": np.abs(observed_freq - predicted_q)
    })

    summary_df = pd.DataFrame([{
        "Model": model_name,
        "Mean_Calibration_Error (MCE)": mce,
        "RMS_Calibration_Error (RMSCE)": rmsce,
        "Num_Quantiles": n_quantiles
    }])

    # -------------------------------
    # Create calibration plot
    # -------------------------------
    fig, ax = plt.subplots(figsize=(7, 7))

    ax.plot([0, 1], [0, 1], 'k--', label="Perfect Calibration")
    ax.plot(
        predicted_q,
        observed_freq,
        'o-',
        color='teal',
        label=f"{model_name}"
    )

    ax.set_xlabel("Predicted Quantile (Confidence Level)")
    ax.set_ylabel("Observed Frequency")
    ax.set_title(f"Quantile Calibration Curve – {model_name}")
    ax.legend()
    fig.tight_layout()

    img_buffer = BytesIO()
    fig.savefig(img_buffer, format="png", dpi=300)
    plt.close(fig)
    img_buffer.seek(0)

    # -------------------------------
    # Save to Excel
    # -------------------------------
    excel_path = Path(excel_path)
    excel_path.parent.mkdir(parents=True, exist_ok=True)

    sheet_curve = f"Calibration_Curve_{model_name}"
    sheet_summary = f"Calibration_Summary_{model_name}"
    sheet_plot = f"Calibration_Plot_{model_name}"

    with pd.ExcelWriter(_ensure_parent_dir(_ensure_excel_file(
        excel_path)),
        engine="openpyxl",
        mode="a" if excel_path.exists() else "w",
        if_sheet_exists="replace"
    ) as writer:

        writer
        calib_df.to_excel(writer, sheet_name=sheet_curve, index=False)
        writer
        summary_df.to_excel(writer, sheet_name=sheet_summary, index=False)

        workbook = writer.book

        worksheet = workbook.create_sheet(sheet_plot)

        img = openpyxlImage(img_buffer)
        img.width = img.width * 0.4
        img.height = img.height * 0.4

        img.anchor = "B2"
        worksheet.add_image(img)

    print("✅ Calibration curve & metrics saved to Excel:")
    print(f"   {excel_path}")

    return calib_df

In [17]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
# Define the path to the folder
folder_path = "/content/drive/MyDrive/Regression/noise_uncertainty_analysis"

# Create the folder if it doesn't exist
os.makedirs(folder_path, exist_ok=True)

def plot_metrics_and_save_to_excel(writer, predictions_df, y_true, model_name):
    # Calculate metrics
    mean_pred = predictions_df['Mean']
    std_pred = predictions_df['StdDev']

    # Sharpness is the standard deviation
    sharpness = std_pred.mean()

    # CRPS for each prediction
    crps_values = ps.crps_gaussian(y_true, mean_pred, std_pred)

    # Negative Log Likelihood (NLL) for each prediction
    nll_values = -norm.logpdf(y_true, loc=mean_pred, scale=std_pred)
    nll = nll_values.mean()

    # Calibration (Validity)
    percentiles = np.linspace(0, 100, 101)
    calibration = [np.mean((y_true <= np.percentile(mean_pred + std_pred * np.random.randn(*mean_pred.shape), p))) for p in percentiles]

    # Create a DataFrame to store the metrics
    metrics_df = pd.DataFrame({
        'Mean Prediction': mean_pred,
        'Standard Deviation': std_pred,
        'CRPS': crps_values,
        'NLL': nll_values
    })

    # Add summary statistics
    summary_df = pd.DataFrame({
        'Metric': ['Sharpness', 'Mean CRPS', 'Mean NLL'],
        'Value': [sharpness, crps_values.mean(), nll]
    })

    # Write the metrics and summary to the Excel sheet
    metrics_sheet_name = f'{model_name} Metrics'
    writer
    metrics_df.to_excel(writer, sheet_name=metrics_sheet_name, index=False)

    summary_sheet_name = f'{model_name} Summary'
    writer
    summary_df.to_excel(writer, sheet_name=summary_sheet_name, index=False)

    # Plotting
    fig, axs = plt.subplots(2, 2, figsize=(12, 10))

    # Sharpness plot
    axs[0, 0].hist(std_pred, bins=30, color='skyblue', edgecolor='black')
    axs[0, 0].set_title(f'Sharpness (StdDev) for {model_name}')
    axs[0, 0].set_xlabel('Standard Deviation')
    axs[0, 0].set_ylabel('Frequency')

    # CRPS plot (distribution)
    axs[0, 1].hist(crps_values, bins=30, color='lightgreen', edgecolor='black')
    axs[0, 1].set_title(f'CRPS Distribution for {model_name}')
    axs[0, 1].set_xlabel('CRPS')
    axs[0, 1].set_ylabel('Frequency')

    # NLL plot (distribution)
    axs[1, 0].hist(nll_values, bins=30, color='salmon', edgecolor='black')
    axs[1, 0].set_title(f'NLL Distribution for {model_name}')
    axs[1, 0].set_xlabel('NLL')
    axs[1, 0].set_ylabel('Frequency')

    # Calibration plot
    axs[1, 1].plot(percentiles, calibration, label='Calibration Curve')
    axs[1, 1].plot(percentiles, percentiles / 100, linestyle='--', label='Perfect Calibration')
    axs[1, 1].set_title(f'Calibration (Validity) for {model_name}')
    axs[1, 1].set_xlabel('Predicted Percentile')
    axs[1, 1].set_ylabel('Observed Frequency')
    axs[1, 1].legend()

    plt.tight_layout()

    # Save plots to images
    plot_image_path = f'{model_name}_plots.png'
    fig.savefig(_ensure_parent_dir(plot_image_path))
    plt.close(fig)

    # Insert the plot image into the Excel sheet
    sheet_name = f'{model_name} Plots'
    if sheet_name not in writer.sheets:  # Check if sheet already exists
        # If sheet doesn't exist, create a new one
        worksheet = writer.book.add_worksheet(sheet_name)
    else:
        # If sheet exists, get the existing sheet object to avoid duplicates
        worksheet = writer.sheets[sheet_name]
    worksheet.insert_image('A1', plot_image_path)

# **Class CARD**

In [18]:
# FIX C1: CARD (Han, Zheng & Zhou, NeurIPS 2022) following the official code,
# github.com/XzwHan/CARD (regression/: diffusion_utils.py, model.py, card_regression.py,
# configs/uci_*.yml). Same class name and API as before: CARDRegressor(base_model, ...),
# .fit(X, y), .predict(X) -> (mean, std).
# Was: the network only saw [X, t] and predicted pure noise (it never saw the noisy target or the
# guide's prediction), there was no noise schedule and only one sampling step, so the output was
# base_pred + N(0, 1) noise for every dataset.
import copy


class _ConditionalLinear(torch.nn.Module):
    # official model.ConditionalLinear: linear layer scaled by a learned per-timestep embedding
    def __init__(self, num_in, num_out, n_steps):
        super().__init__()
        self.num_out = num_out
        self.lin = torch.nn.Linear(num_in, num_out)
        self.embed = torch.nn.Embedding(n_steps, num_out)
        self.embed.weight.data.uniform_()

    def forward(self, x, t):
        return self.embed(t).view(-1, self.num_out) * self.lin(x)


class _ConditionalGuidedModel(torch.nn.Module):
    # official model.ConditionalGuidedModel with cat_x=True, cat_y_pred=True: eps_theta(x, y_t, y_0_hat, t)
    def __init__(self, x_dim, hidden_dim, n_steps):
        super().__init__()
        self.lin1 = _ConditionalLinear(x_dim + 2, hidden_dim, n_steps)
        self.lin2 = _ConditionalLinear(hidden_dim, hidden_dim, n_steps)
        self.lin3 = _ConditionalLinear(hidden_dim, hidden_dim, n_steps)
        self.lin4 = torch.nn.Linear(hidden_dim, 1)

    def forward(self, x, y_t, y_0_hat, t):
        h = torch.cat((y_t, y_0_hat, x), dim=1)
        h = torch.nn.functional.softplus(self.lin1(h, t))
        h = torch.nn.functional.softplus(self.lin2(h, t))
        h = torch.nn.functional.softplus(self.lin3(h, t))
        return self.lin4(h)


class CARDRegressor:
    def __init__(
        self,
        base_model,
        hidden_dim=128,
        T=1000,            # official configs/uci_*.yml: diffusion.timesteps
        lr=1e-3,           # official optim.lr
        epochs=5000,       # official training.n_epochs
        n_samples=1000,    # official testing.n_z_samples
        device="cpu",
        batch_size=32,     # official training.batch_size
        beta_start=1e-4,   # official linear beta schedule
        beta_end=0.02,
        ema_rate=0.9999,   # official model.ema_rate (EMA weights are used for sampling)
        ema_warmup=True,   # EMA decay min(ema_rate, (1+k)/(10+k)); see note in fit()
        seed=42,
    ):
        self.base_model = base_model
        self.hidden_dim = hidden_dim
        self.T = T
        self.lr = lr
        self.epochs = epochs
        self.n_samples = n_samples
        self.device = device
        self.batch_size = batch_size
        self.beta_start = beta_start
        self.beta_end = beta_end
        self.ema_rate = ema_rate
        self.ema_warmup = ema_warmup
        self.seed = seed
        self.mlp = None

    # ---- helpers -------------------------------------------------------------
    def _guide(self, X):
        # pre-trained guidance model f_phi(x), in normalized target units
        p = self.base_model.predict(X)
        if torch.is_tensor(p):
            p = p.detach().cpu().numpy()
        p = np.asarray(p, dtype=np.float64).ravel()
        return (p - self.y_mean_) / self.y_std_

    def _schedule(self):
        betas = torch.linspace(self.beta_start, self.beta_end, self.T)   # make_beta_schedule("linear")
        alphas = 1.0 - betas
        alphas_cumprod = torch.cumprod(alphas, dim=0)
        self.alphas = alphas
        self.alphas_bar_sqrt = torch.sqrt(alphas_cumprod)
        self.one_minus_alphas_bar_sqrt = torch.sqrt(1.0 - alphas_cumprod)

    # ---- training: noise-estimation loss (card_regression.py) -----------------
    def fit(self, X, y):
        torch.manual_seed(self.seed)
        rng = np.random.default_rng(self.seed)
        X = np.asarray(X, dtype=np.float64)
        y = np.asarray(y, dtype=np.float64).ravel()
        self.y_mean_, self.y_std_ = float(y.mean()), float(y.std() + 1e-12)   # normalize_y: True
        y_n = (y - self.y_mean_) / self.y_std_
        y0_hat = self._guide(X)
        self._schedule()

        X_t = torch.tensor(X, dtype=torch.float32, device=self.device)
        y_t0 = torch.tensor(y_n, dtype=torch.float32, device=self.device).view(-1, 1)
        g_t0 = torch.tensor(y0_hat, dtype=torch.float32, device=self.device).view(-1, 1)
        abs_ = self.alphas_bar_sqrt.to(self.device)
        omabs = self.one_minus_alphas_bar_sqrt.to(self.device)

        self.mlp = _ConditionalGuidedModel(X.shape[1], self.hidden_dim, self.T + 1).to(self.device)
        optimizer = torch.optim.Adam(self.mlp.parameters(), lr=self.lr)
        ema = {k: v.detach().clone() for k, v in self.mlp.state_dict().items()}

        n = len(X_t)
        step = 0
        for _ in range(self.epochs):
            perm = rng.permutation(n)
            for start in range(0, n, self.batch_size):
                idx = torch.as_tensor(perm[start:start + self.batch_size], device=self.device)
                xb, yb, gb = X_t[idx], y_t0[idx], g_t0[idx]
                m = len(idx)
                # antithetic sampling of timesteps
                t = torch.randint(0, self.T, (m // 2 + 1,), device=self.device)
                t = torch.cat([t, self.T - 1 - t], dim=0)[:m]
                e = torch.randn_like(yb)
                # q_sample: y_t = sqrt(abar_t) y_0 + (1 - sqrt(abar_t)) y_0_hat + sqrt(1 - abar_t) e
                a_t = abs_[t].view(-1, 1)
                y_tb = a_t * yb + (1.0 - a_t) * gb + omabs[t].view(-1, 1) * e
                loss = (e - self.mlp(xb, y_tb, gb, t)).square().mean()
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                # EMA of the weights (official: ema_rate 0.9999). With warm-up, the decay grows from 0
                # towards ema_rate as training proceeds, so the averaged weights never stay dominated
                # by the random initial weights when the number of updates is small (generic for any
                # dataset size; with 100k+ updates it is the same as the official EMA).
                step += 1
                decay = min(self.ema_rate, (1.0 + step) / (10.0 + step)) if self.ema_warmup else self.ema_rate
                with torch.no_grad():
                    for k, v in self.mlp.state_dict().items():
                        if v.dtype.is_floating_point:
                            ema[k].mul_(decay).add_(v, alpha=1.0 - decay)
        self.mlp_ema_ = copy.deepcopy(self.mlp)
        self.mlp_ema_.load_state_dict(ema)
        return self

    # ---- sampling: p_sample_loop (diffusion_utils.py) -------------------------
    @torch.no_grad()
    def sample(self, X):
        X = np.asarray(X, dtype=np.float64)
        model = self.mlp_ema_
        n = len(X)
        S = self.n_samples
        x = torch.tensor(X, dtype=torch.float32, device=self.device).repeat_interleave(S, dim=0)
        g = torch.tensor(self._guide(X), dtype=torch.float32, device=self.device).view(-1, 1).repeat_interleave(S, dim=0)
        alphas = self.alphas.to(self.device)
        omabs = self.one_minus_alphas_bar_sqrt.to(self.device)
        y = g + torch.randn_like(g)                     # y_T ~ N(y_0_hat, I)
        for t in reversed(range(1, self.T)):            # t = T-1 ... 1  -> y_{t-1}
            tt = torch.full((len(y),), t, dtype=torch.long, device=self.device)
            alpha_t = alphas[t]
            s1m_t, s1m_tm1 = omabs[t], omabs[t - 1]
            sab_t = (1 - s1m_t ** 2).sqrt()
            sab_tm1 = (1 - s1m_tm1 ** 2).sqrt()
            gamma_0 = (1 - alpha_t) * sab_tm1 / (s1m_t ** 2)
            gamma_1 = (s1m_tm1 ** 2) * alpha_t.sqrt() / (s1m_t ** 2)
            gamma_2 = 1 + (sab_t - 1) * (alpha_t.sqrt() + sab_tm1) / (s1m_t ** 2)
            eps = model(x, y, g, tt)
            y0_reparam = (y - (1 - sab_t) * g - eps * s1m_t) / sab_t
            y_mean = gamma_0 * y0_reparam + gamma_1 * y + gamma_2 * g
            beta_t_hat = (s1m_tm1 ** 2) / (s1m_t ** 2) * (1 - alpha_t)
            y = y_mean + beta_t_hat.sqrt() * torch.randn_like(y)
        # final step t = 1 -> 0 (p_sample_t_1to0)
        tt = torch.zeros(len(y), dtype=torch.long, device=self.device)
        s1m_0 = omabs[0]
        sab_0 = (1 - s1m_0 ** 2).sqrt()
        eps = model(x, y, g, tt)
        y0 = (y - (1 - sab_0) * g - eps * s1m_0) / sab_0
        y0 = y0.view(n, S).cpu().numpy().astype(np.float64)
        return y0 * self.y_std_ + self.y_mean_          # back to original target units

    def predict(self, X):
        samples = self.sample(X)
        return samples.mean(axis=1), samples.std(axis=1)


# **Tuned Parameters**

In [19]:
# FIX K2: load the tuning results saved by hyperparameter_tuning/Optuna_autosampler.ipynb
# (was: a hand-pasted copy of an older tuning run that had been selected on the TEST set).
import pickle
if IN_KAGGLE:
    _bp = glob.glob("/kaggle/input/**/hyperparameter_tuning/best_params.pkl", recursive=True)[0]
else:
    _bp = "./drive/MyDrive/Regression/noise_uncertainty_analysis/hyperparameter_tuning/best_params.pkl"
with open(_bp, "rb") as f:
    best_scores_autosampler = pickle.load(f)
print("Loaded tuning results from", _bp, "-", len(best_scores_autosampler), "model x pruner entries")


Loaded tuning results from /kaggle/input/datasets/daneshselwal/noise-tuned-params/hyperparameter_tuning/best_params.pkl - 80 model x pruner entries


# **Probabilistic Distribution using LGBM**

In [20]:
best_params = get_best_model_params(best_scores_autosampler, 'LightGBM')

lgbm_model = LGBMRegressor(**best_params)
lgbm_model.fit(X_train, y_train)

card_model = CARDRegressor(  # FIX C1: official CARD settings (T, epochs, n_z_samples)
    base_model=lgbm_model,
    hidden_dim=128,
    T=1000,
    epochs=5000,
    n_samples=1000
)

card_model.fit(X_train, y_train)

mean, stddev = card_model.predict(X_test)

predictions_LGBM_df = pd.DataFrame({
    'Mean': mean,
    'StdDev': stddev,
    'Actual': y_test
})

print(predictions_LGBM_df.head())


        Mean    StdDev  Actual
0  71.173998  0.349131   70.07
1  73.840916  0.348437   72.90
2  62.179450  0.351354   68.66
3  66.754164  0.344900   50.24
4  36.444818  0.339609   40.17


In [21]:
generate_and_save_plots(predictions_LGBM_df, y_test, "./drive/MyDrive/Regression/noise_uncertainty_analysis/probabilistic_distribution_card/LightGBM.xlsx")

Excel file saved at ./drive/MyDrive/Regression/noise_uncertainty_analysis/probabilistic_distribution_card/LightGBM.xlsx


In [22]:
generate_and_save_crps_decomposition(
    predictions_LGBM_df,
    y_test,
    "./drive/MyDrive/Regression/noise_uncertainty_analysis/probabilistic_distribution_card/LightGBM.xlsx",
    model_name="LightGBM",
    n_bins=N_BINS
)

✅ CRPS decomposition saved to:
drive/MyDrive/Regression/noise_uncertainty_analysis/probabilistic_distribution_card/LightGBM.xlsx
   Sheets: CRPS_Summary_LightGBM, CRPS_Bins_LightGBM


,Model,Total_CRPS,Reliability,Resolution,Uncertainty,Num_Bins
0,LightGBM,2.461563,0.848438,6.265431,7.879017,6


In [23]:
generate_and_save_pit_results(
    predictions_LGBM_df,
    y_test,
    "./drive/MyDrive/Regression/noise_uncertainty_analysis/probabilistic_distribution_card/LightGBM.xlsx",
    model_name="LightGBM",
    n_bins=N_BINS
)

✅ PIT values, histogram, summary, and EMBEDDED plot saved to:
   drive/MyDrive/Regression/noise_uncertainty_analysis/probabilistic_distribution_card/LightGBM.xlsx


,PIT_Value
0,7.830976e-04
1,3.462914e-03
2,1.000000e+00
3,0.000000e+00
4,1.000000e+00
5,1.000000e+00
6,3.951031e-01
7,3.030221e-03
8,8.316880e-11
9,1.000000e+00


In [24]:
generate_and_save_calibration_curve(
    predictions_LGBM_df,
    y_test,
    "./drive/MyDrive/Regression/noise_uncertainty_analysis/probabilistic_distribution_card/LightGBM.xlsx",
    model_name="LightGBM",
    n_quantiles=10
)

✅ Calibration curve & metrics saved to Excel:
   drive/MyDrive/Regression/noise_uncertainty_analysis/probabilistic_distribution_card/LightGBM.xlsx


,Predicted_Quantile,Observed_Frequency,Absolute_Error
0,0.0,0.000000,0.000000
1,0.1,0.433333,0.333333
2,0.2,0.466667,0.266667
3,0.3,0.466667,0.166667
4,0.4,0.500000,0.100000
5,0.5,0.500000,0.000000
6,0.6,0.516667,0.083333
7,0.7,0.533333,0.166667
8,0.8,0.583333,0.216667
9,0.9,0.583333,0.316667


# **Probabilistic Distribution using XGBoost**

In [25]:
best_params = get_best_model_params(best_scores_autosampler, 'XGBoost')

xgb_model = XGBRegressor(**best_params)
xgb_model.fit(X_train, y_train)

card_model = CARDRegressor(  # FIX C1: official CARD settings (T, epochs, n_z_samples)
    base_model=xgb_model,
    hidden_dim=128,
    T=1000,
    epochs=5000,
    n_samples=1000
)

card_model.fit(X_train, y_train)

mean, stddev = card_model.predict(X_test)

predictions_XGB_df = pd.DataFrame({
    'Mean': mean,
    'StdDev': stddev,
    'Actual': y_test
})

print(predictions_XGB_df.head())

        Mean    StdDev  Actual
0  72.479479  0.997284   70.07
1  76.207111  0.971790   72.90
2  57.946129  0.977789   68.66
3  62.929157  0.991806   50.24
4  37.483335  0.975614   40.17


In [26]:
generate_and_save_plots(predictions_XGB_df, y_test, "./drive/MyDrive/Regression/noise_uncertainty_analysis/probabilistic_distribution_card/XGBoost.xlsx")

Excel file saved at ./drive/MyDrive/Regression/noise_uncertainty_analysis/probabilistic_distribution_card/XGBoost.xlsx


In [27]:
generate_and_save_crps_decomposition(
    predictions_XGB_df,
    y_test,
    "./drive/MyDrive/Regression/noise_uncertainty_analysis/probabilistic_distribution_card/XGBoost.xlsx",
    model_name="XGBoost",
    n_bins=N_BINS
)

✅ CRPS decomposition saved to:
drive/MyDrive/Regression/noise_uncertainty_analysis/probabilistic_distribution_card/XGBoost.xlsx
   Sheets: CRPS_Summary_XGBoost, CRPS_Bins_XGBoost


,Model,Total_CRPS,Reliability,Resolution,Uncertainty,Num_Bins
0,XGBoost,2.300642,0.421061,5.998811,7.879017,6


In [28]:
generate_and_save_pit_results(
    predictions_XGB_df,
    y_test,
    "./drive/MyDrive/Regression/noise_uncertainty_analysis/probabilistic_distribution_card/XGBoost.xlsx",
    model_name="XGBoost",
    n_bins=N_BINS
)

✅ PIT values, histogram, summary, and EMBEDDED plot saved to:
   drive/MyDrive/Regression/noise_uncertainty_analysis/probabilistic_distribution_card/XGBoost.xlsx


,PIT_Value
0,7.845143e-03
1,3.331134e-04
2,1.000000e+00
3,8.856286e-38
4,9.970548e-01
5,9.855525e-01
6,1.540234e-01
7,6.103017e-01
8,1.001435e-03
9,6.261700e-01


In [29]:
generate_and_save_calibration_curve(
    predictions_XGB_df,
    y_test,
    "./drive/MyDrive/Regression/noise_uncertainty_analysis/probabilistic_distribution_card/XGBoost.xlsx",
    model_name="XGBoost",
    n_quantiles=10
)

✅ Calibration curve & metrics saved to Excel:
   drive/MyDrive/Regression/noise_uncertainty_analysis/probabilistic_distribution_card/XGBoost.xlsx


,Predicted_Quantile,Observed_Frequency,Absolute_Error
0,0.0,0.000000,0.000000
1,0.1,0.400000,0.300000
2,0.2,0.466667,0.266667
3,0.3,0.500000,0.200000
4,0.4,0.550000,0.150000
5,0.5,0.600000,0.100000
6,0.6,0.616667,0.016667
7,0.7,0.683333,0.016667
8,0.8,0.750000,0.050000
9,0.9,0.800000,0.100000


# **Probabilistic Distribution using GPBoost**


In [30]:
best_params = get_best_model_params(best_scores_autosampler, 'GPBoost')

gpboost_model = GPBoostRegressor(**best_params)
gpboost_model.fit(X_train, y_train)

card_model = CARDRegressor(  # FIX C1: official CARD settings (T, epochs, n_z_samples)
    base_model=gpboost_model,
    hidden_dim=128,
    T=1000,
    epochs=5000,
    n_samples=1000
)


card_model.fit(X_train, y_train)

mean, stddev = card_model.predict(X_test)

predictions_GPBoost_df = pd.DataFrame({
    'Mean': mean,
    'StdDev': stddev,
    'Actual': y_test
})

print(predictions_GPBoost_df.head())

        Mean    StdDev  Actual
0  71.154858  0.512913   70.07
1  74.781490  0.508956   72.90
2  65.895257  0.511447   68.66
3  63.714743  0.512694   50.24
4  37.611590  0.505490   40.17


In [31]:
generate_and_save_plots(predictions_GPBoost_df, y_test, "./drive/MyDrive/Regression/noise_uncertainty_analysis/probabilistic_distribution_card/GPBoost.xlsx")

Excel file saved at ./drive/MyDrive/Regression/noise_uncertainty_analysis/probabilistic_distribution_card/GPBoost.xlsx


In [32]:
generate_and_save_crps_decomposition(
    predictions_GPBoost_df,
    y_test,
    "./drive/MyDrive/Regression/noise_uncertainty_analysis/probabilistic_distribution_card/GPBoost.xlsx",
    model_name="GPBoost",
    n_bins=N_BINS
)

✅ CRPS decomposition saved to:
drive/MyDrive/Regression/noise_uncertainty_analysis/probabilistic_distribution_card/GPBoost.xlsx
   Sheets: CRPS_Summary_GPBoost, CRPS_Bins_GPBoost


,Model,Total_CRPS,Reliability,Resolution,Uncertainty,Num_Bins
0,GPBoost,1.99652,0.479354,6.361319,7.879017,6


In [33]:
generate_and_save_pit_results(
    predictions_GPBoost_df,
    y_test,
    "./drive/MyDrive/Regression/noise_uncertainty_analysis/probabilistic_distribution_card/GPBoost.xlsx",
    model_name="GPBoost",
    n_bins=N_BINS
)

✅ PIT values, histogram, summary, and EMBEDDED plot saved to:
   drive/MyDrive/Regression/noise_uncertainty_analysis/probabilistic_distribution_card/GPBoost.xlsx


,PIT_Value
0,1.721113e-02
1,1.091843e-04
2,1.000000e+00
3,1.531417e-152
4,9.999998e-01
5,9.999930e-01
6,2.473488e-02
7,2.682369e-04
8,7.722464e-04
9,1.000000e+00


In [34]:
generate_and_save_calibration_curve(
    predictions_GPBoost_df,
    y_test,
    "./drive/MyDrive/Regression/noise_uncertainty_analysis/probabilistic_distribution_card/GPBoost.xlsx",
    model_name="GPBoost",
    n_quantiles=10
)

✅ Calibration curve & metrics saved to Excel:
   drive/MyDrive/Regression/noise_uncertainty_analysis/probabilistic_distribution_card/GPBoost.xlsx


,Predicted_Quantile,Observed_Frequency,Absolute_Error
0,0.0,0.000000,0.000000
1,0.1,0.350000,0.250000
2,0.2,0.383333,0.183333
3,0.3,0.466667,0.166667
4,0.4,0.483333,0.083333
5,0.5,0.516667,0.016667
6,0.6,0.566667,0.033333
7,0.7,0.600000,0.100000
8,0.8,0.616667,0.183333
9,0.9,0.633333,0.266667


# **Probabilistic distributions using NGBoost**

In [35]:

best_params = get_best_model_params(best_scores_autosampler, 'NGBoost')

incompatible_keys = ['Dist', 'Score']
for key in incompatible_keys:
    best_params.pop(key, None)

ngb_model = NGBRegressor(Dist=Normal, **best_params)

ngb_model.fit(X_train, y_train)

card_model = CARDRegressor(  # FIX C1: official CARD settings (T, epochs, n_z_samples)
    base_model=ngb_model,
    hidden_dim=128,
    T=1000,
    epochs=5000,
    n_samples=1000
)

card_model.fit(X_train, y_train)

mean_pred, std_pred = card_model.predict(X_test)


predictions_NGB_df = pd.DataFrame({
    "Mean": mean_pred,
    "StdDev": std_pred,
    "Actual": y_test.ravel()
})

mse = mean_squared_error(y_test, mean_pred)
print(f"Mean Squared Error: {mse:.2f}")

print(predictions_NGB_df.head())

Mean Squared Error: 19.34
        Mean    StdDev  Actual
0  69.451902  1.924122   70.07
1  76.534469  1.674357   72.90
2  57.206367  1.868500   68.66
3  63.911294  1.967297   50.24
4  37.808108  2.005318   40.17


In [36]:
generate_and_save_plots(predictions_NGB_df, y_test, "./drive/MyDrive/Regression/noise_uncertainty_analysis/probabilistic_distribution_card/NGBoost.xlsx")

Excel file saved at ./drive/MyDrive/Regression/noise_uncertainty_analysis/probabilistic_distribution_card/NGBoost.xlsx


In [37]:
generate_and_save_crps_decomposition(
    predictions_NGB_df,
    y_test,
    "./drive/MyDrive/Regression/noise_uncertainty_analysis/probabilistic_distribution_card/NGBoost.xlsx",
    model_name="NGBoost",
    n_bins=N_BINS
)

✅ CRPS decomposition saved to:
drive/MyDrive/Regression/noise_uncertainty_analysis/probabilistic_distribution_card/NGBoost.xlsx
   Sheets: CRPS_Summary_NGBoost, CRPS_Bins_NGBoost


,Model,Total_CRPS,Reliability,Resolution,Uncertainty,Num_Bins
0,NGBoost,2.378749,0.172274,5.671803,7.879017,6


In [38]:
generate_and_save_pit_results(
    predictions_NGB_df,
    y_test,
    "./drive/MyDrive/Regression/noise_uncertainty_analysis/probabilistic_distribution_card/NGBoost.xlsx",
    model_name="NGBoost",
    n_bins=N_BINS
)

✅ PIT values, histogram, summary, and EMBEDDED plot saved to:
   drive/MyDrive/Regression/noise_uncertainty_analysis/probabilistic_distribution_card/NGBoost.xlsx


,PIT_Value
0,6.259844e-01
1,1.497825e-02
2,1.000000e+00
3,1.835804e-12
4,8.805647e-01
5,9.756170e-01
6,3.780495e-01
7,1.482348e-01
8,2.993309e-03
9,9.999892e-01


In [39]:
generate_and_save_calibration_curve(
    predictions_NGB_df,
    y_test,
    "./drive/MyDrive/Regression/noise_uncertainty_analysis/probabilistic_distribution_card/NGBoost.xlsx",
    model_name="NGBoost",
    n_quantiles=10
)

✅ Calibration curve & metrics saved to Excel:
   drive/MyDrive/Regression/noise_uncertainty_analysis/probabilistic_distribution_card/NGBoost.xlsx


,Predicted_Quantile,Observed_Frequency,Absolute_Error
0,0.0,0.000000,0.000000
1,0.1,0.200000,0.100000
2,0.2,0.300000,0.100000
3,0.3,0.400000,0.100000
4,0.4,0.500000,0.100000
5,0.5,0.600000,0.100000
6,0.6,0.683333,0.083333
7,0.7,0.750000,0.050000
8,0.8,0.750000,0.050000
9,0.9,0.816667,0.083333


# **Probabilistic Distribution using Gradient Boosting**

In [40]:
best_params = get_best_model_params(best_scores_autosampler, 'GradientBoosting')

gbr_model = GradientBoostingRegressor(**best_params)
gbr_model.fit(X_train, y_train)

card_model = CARDRegressor(  # FIX C1: official CARD settings (T, epochs, n_z_samples)
    base_model=gbr_model,
    hidden_dim=128,
    T=1000,
    epochs=5000,
    n_samples=1000
)

card_model.fit(X_train, y_train)

mean, stddev = card_model.predict(X_test)

predictions_GBR_df = pd.DataFrame({
    'Mean': mean,
    'StdDev': stddev,
    'Actual': y_test
})

print(predictions_GBR_df.head())

        Mean    StdDev  Actual
0  71.976390  2.330672   70.07
1  75.863056  1.517374   72.90
2  53.713148  2.284180   68.66
3  59.500767  4.110061   50.24
4  36.757772  4.026786   40.17


In [41]:
generate_and_save_plots(predictions_GBR_df, y_test, "./drive/MyDrive/Regression/noise_uncertainty_analysis/probabilistic_distribution_card/Gradient Boosting.xlsx")

Excel file saved at ./drive/MyDrive/Regression/noise_uncertainty_analysis/probabilistic_distribution_card/Gradient Boosting.xlsx


In [42]:
generate_and_save_crps_decomposition(
    predictions_GBR_df,
    y_test,
    "./drive/MyDrive/Regression/noise_uncertainty_analysis/probabilistic_distribution_card/Gradient Boosting.xlsx",
    model_name="Gradient Boosting",
    n_bins=N_BINS
)

✅ CRPS decomposition saved to:
drive/MyDrive/Regression/noise_uncertainty_analysis/probabilistic_distribution_card/Gradient Boosting.xlsx
   Sheets: CRPS_Summary_Gradient Boosting, CRPS_Bins_Gradient Boosting


,Model,Total_CRPS,Reliability,Resolution,Uncertainty,Num_Bins
0,Gradient Boosting,1.944597,0.109456,6.043666,7.879017,6


In [43]:
generate_and_save_pit_results(
    predictions_GBR_df,
    y_test,
    "./drive/MyDrive/Regression/noise_uncertainty_analysis/probabilistic_distribution_card/Gradient Boosting.xlsx",
    model_name="GBoosting",
    n_bins=N_BINS
)

✅ PIT values, histogram, summary, and EMBEDDED plot saved to:
   drive/MyDrive/Regression/noise_uncertainty_analysis/probabilistic_distribution_card/Gradient Boosting.xlsx


,PIT_Value
0,0.206691
1,0.025424
2,1.000000
3,0.012123
4,0.801609
5,0.987534
6,0.587128
7,0.304563
8,0.150424
9,0.526061


In [44]:
generate_and_save_calibration_curve(
    predictions_GBR_df,
    y_test,
    "./drive/MyDrive/Regression/noise_uncertainty_analysis/probabilistic_distribution_card/Gradient Boosting.xlsx",
    model_name="GBoosting",
    n_quantiles=10
)

✅ Calibration curve & metrics saved to Excel:
   drive/MyDrive/Regression/noise_uncertainty_analysis/probabilistic_distribution_card/Gradient Boosting.xlsx


,Predicted_Quantile,Observed_Frequency,Absolute_Error
0,0.0,0.000000,0.000000
1,0.1,0.133333,0.033333
2,0.2,0.250000,0.050000
3,0.3,0.383333,0.083333
4,0.4,0.466667,0.066667
5,0.5,0.583333,0.083333
6,0.6,0.716667,0.116667
7,0.7,0.816667,0.116667
8,0.8,0.866667,0.066667
9,0.9,0.933333,0.033333


# **Probabilistic Distribution using Catboost**

In [45]:

best_params = get_best_model_params(best_scores_autosampler, 'CatBoost')

catboost_model = CatBoostRegressor(**best_params)
catboost_model.fit(X_train, y_train)

card_model = CARDRegressor(  # FIX C1: official CARD settings (T, epochs, n_z_samples)
    base_model=catboost_model,
    hidden_dim=128,
    T=1000,
    epochs=5000,
    n_samples=1000
)

card_model.fit(X_train, y_train)

mean, stddev = card_model.predict(X_test)

predictions_CatBoost_df = pd.DataFrame({
    'Mean': mean,
    'StdDev': stddev,
    'Actual': y_test
})

print(predictions_CatBoost_df.head())


        Mean    StdDev  Actual
0  72.253860  1.923493   70.07
1  76.027953  1.606026   72.90
2  61.097892  1.814792   68.66
3  67.310395  2.022156   50.24
4  37.047440  2.015432   40.17


In [46]:
generate_and_save_plots(predictions_CatBoost_df, y_test, "./drive/MyDrive/Regression/noise_uncertainty_analysis/probabilistic_distribution_card/CatBoost.xlsx")

Excel file saved at ./drive/MyDrive/Regression/noise_uncertainty_analysis/probabilistic_distribution_card/CatBoost.xlsx


In [47]:
generate_and_save_crps_decomposition(
    predictions_CatBoost_df,
    y_test,
    "./drive/MyDrive/Regression/noise_uncertainty_analysis/probabilistic_distribution_card/CatBoost.xlsx",
    model_name="CatBoost",
    n_bins=N_BINS
)


✅ CRPS decomposition saved to:
drive/MyDrive/Regression/noise_uncertainty_analysis/probabilistic_distribution_card/CatBoost.xlsx
   Sheets: CRPS_Summary_CatBoost, CRPS_Bins_CatBoost


,Model,Total_CRPS,Reliability,Resolution,Uncertainty,Num_Bins
0,CatBoost,2.251229,0.121636,5.748816,7.879017,6


In [48]:
generate_and_save_pit_results(
    predictions_CatBoost_df,
    y_test,
    "./drive/MyDrive/Regression/noise_uncertainty_analysis/probabilistic_distribution_card/CatBoost.xlsx",
    model_name="CatBoost",
    n_bins=N_BINS
)

✅ PIT values, histogram, summary, and EMBEDDED plot saved to:
   drive/MyDrive/Regression/noise_uncertainty_analysis/probabilistic_distribution_card/CatBoost.xlsx


,PIT_Value
0,1.281119e-01
1,2.572929e-02
2,9.999846e-01
3,1.564042e-17
4,9.393482e-01
5,9.894539e-01
6,3.459141e-01
7,4.460380e-01
8,3.737016e-03
9,9.907038e-01


In [49]:
generate_and_save_calibration_curve(
    predictions_CatBoost_df,
    y_test,
    "./drive/MyDrive/Regression/noise_uncertainty_analysis/probabilistic_distribution_card/CatBoost.xlsx",
    model_name="CatBoost",
    n_quantiles=10
)

✅ Calibration curve & metrics saved to Excel:
   drive/MyDrive/Regression/noise_uncertainty_analysis/probabilistic_distribution_card/CatBoost.xlsx


,Predicted_Quantile,Observed_Frequency,Absolute_Error
0,0.0,0.000000,0.000000e+00
1,0.1,0.166667,6.666667e-02
2,0.2,0.250000,5.000000e-02
3,0.3,0.416667,1.166667e-01
4,0.4,0.533333,1.333333e-01
5,0.5,0.616667,1.166667e-01
6,0.6,0.650000,5.000000e-02
7,0.7,0.700000,1.110223e-16
8,0.8,0.750000,5.000000e-02
9,0.9,0.833333,6.666667e-02


# **Probabilistic Distribution using HGBM**

In [50]:


best_params = get_best_model_params(best_scores_autosampler, 'HGBR')


hgb_model = HistGradientBoostingRegressor(
    **best_params,
)
hgb_model.fit(X_train, y_train)

card_model = CARDRegressor(  # FIX C1: official CARD settings (T, epochs, n_z_samples)
    base_model=hgb_model,
    hidden_dim=128,
    T=1000,
    epochs=5000,
    n_samples=1000
)

card_model.fit(X_train, y_train)

mean, stddev = card_model.predict(X_test)

predictions_HGBR_df = pd.DataFrame({
    "Mean": mean,
    "StdDev": stddev,
    "Actual": y_test.values.ravel()
})

print(predictions_HGBR_df.head())


        Mean    StdDev  Actual
0  74.515866  2.155026   70.07
1  76.054205  1.472699   72.90
2  57.126313  2.502643   68.66
3  63.255572  3.015736   50.24
4  35.511994  3.956435   40.17


In [51]:
generate_and_save_plots(predictions_HGBR_df, y_test, "./drive/MyDrive/Regression/noise_uncertainty_analysis/probabilistic_distribution_card/HGBM.xlsx")

Excel file saved at ./drive/MyDrive/Regression/noise_uncertainty_analysis/probabilistic_distribution_card/HGBM.xlsx


In [52]:
generate_and_save_crps_decomposition(
    predictions_HGBR_df,
    y_test,
    "./drive/MyDrive/Regression/noise_uncertainty_analysis/probabilistic_distribution_card/HGBM.xlsx",
    model_name="HGBR",
    n_bins=N_BINS
)


✅ CRPS decomposition saved to:
drive/MyDrive/Regression/noise_uncertainty_analysis/probabilistic_distribution_card/HGBM.xlsx
   Sheets: CRPS_Summary_HGBR, CRPS_Bins_HGBR


,Model,Total_CRPS,Reliability,Resolution,Uncertainty,Num_Bins
0,HGBR,2.027343,0.160742,6.012051,7.879017,6


In [53]:
generate_and_save_pit_results(
    predictions_HGBR_df,
    y_test,
    "./drive/MyDrive/Regression/noise_uncertainty_analysis/probabilistic_distribution_card/HGBM.xlsx",
    model_name="HGBR",
    n_bins=N_BINS
)

✅ PIT values, histogram, summary, and EMBEDDED plot saved to:
   drive/MyDrive/Regression/noise_uncertainty_analysis/probabilistic_distribution_card/HGBM.xlsx


,PIT_Value
0,0.019555
1,0.016105
2,0.999998
3,0.000008
4,0.880467
5,0.953833
6,0.148798
7,0.178691
8,0.152013
9,0.820171


In [54]:
generate_and_save_calibration_curve(
    predictions_HGBR_df,
    y_test,
    "./drive/MyDrive/Regression/noise_uncertainty_analysis/probabilistic_distribution_card/HGBM.xlsx",
    model_name="HGBR",
    n_quantiles=10
)

✅ Calibration curve & metrics saved to Excel:
   drive/MyDrive/Regression/noise_uncertainty_analysis/probabilistic_distribution_card/HGBM.xlsx


,Predicted_Quantile,Observed_Frequency,Absolute_Error
0,0.0,0.000000,0.000000
1,0.1,0.166667,0.066667
2,0.2,0.333333,0.133333
3,0.3,0.466667,0.166667
4,0.4,0.583333,0.183333
5,0.5,0.666667,0.166667
6,0.6,0.733333,0.133333
7,0.7,0.783333,0.083333
8,0.8,0.816667,0.016667
9,0.9,0.900000,0.000000


# **Probabilistic Distribution using TabNet**

In [55]:
class TabNetMeanWrapper:
    def __init__(self, model):
        self.model = model

    def fit(self, X, y):
        X = np.asarray(X)
        y = np.asarray(y).reshape(-1, 1)

        self.model.fit(
            X,
            y,
            max_epochs=200,
            patience=30,
            batch_size=1024,
            virtual_batch_size=128,
            num_workers=0,
            drop_last=False
        )

    def predict(self, X):
        X = np.asarray(X)
        return self.model.predict(X).reshape(-1)

best_params = get_best_model_params(best_scores_autosampler, 'TabNet')

tabnet_base = TabNetMeanWrapper(
    TabNetRegressor(**best_params)
)

tabnet_base.fit(X_train, y_train)

card_model = CARDRegressor(  # FIX C1: official CARD settings (T, epochs, n_z_samples)
    base_model=tabnet_base,
    hidden_dim=128,
    T=1000,
    epochs=5000,
    n_samples=1000
)

card_model.fit(X_train, y_train)

mean, stddev = card_model.predict(X_test)

predictions_TabNet_df = pd.DataFrame({
    "Mean": mean,
    "StdDev": stddev,
    "Actual": y_test.values
})

print(predictions_TabNet_df.head())


        Mean    StdDev  Actual
0  70.662412  2.467216   70.07
1  74.663359  1.689367   72.90
2  65.457299  2.960857   68.66
3  59.459990  5.052211   50.24
4  37.620007  4.286943   40.17


In [56]:
generate_and_save_plots(predictions_TabNet_df, y_test, "./drive/MyDrive/Regression/noise_uncertainty_analysis/probabilistic_distribution_card/TabNet.xlsx")

Excel file saved at ./drive/MyDrive/Regression/noise_uncertainty_analysis/probabilistic_distribution_card/TabNet.xlsx


In [57]:
generate_and_save_crps_decomposition(
    predictions_TabNet_df,
    y_test,
    "./drive/MyDrive/Regression/noise_uncertainty_analysis/probabilistic_distribution_card/TabNet.xlsx",
    model_name="TabNet",
    n_bins=N_BINS
)


✅ CRPS decomposition saved to:
drive/MyDrive/Regression/noise_uncertainty_analysis/probabilistic_distribution_card/TabNet.xlsx
   Sheets: CRPS_Summary_TabNet, CRPS_Bins_TabNet


,Model,Total_CRPS,Reliability,Resolution,Uncertainty,Num_Bins
0,TabNet,2.191618,0.046649,5.733829,7.879017,6


In [58]:
generate_and_save_pit_results(
    predictions_TabNet_df,
    y_test,
    "./drive/MyDrive/Regression/noise_uncertainty_analysis/probabilistic_distribution_card/TabNet.xlsx",
    model_name="TabNet",
    n_bins=N_BINS
)

✅ PIT values, histogram, summary, and EMBEDDED plot saved to:
   drive/MyDrive/Regression/noise_uncertainty_analysis/probabilistic_distribution_card/TabNet.xlsx


,PIT_Value
0,0.405121
1,0.148289
2,0.860303
3,0.034005
4,0.724021
5,0.985312
6,0.693702
7,0.274555
8,0.111455
9,0.956087


In [59]:
generate_and_save_calibration_curve(
    predictions_TabNet_df,
    y_test,
    "./drive/MyDrive/Regression/noise_uncertainty_analysis/probabilistic_distribution_card/TabNet.xlsx",  # FIX B2: was CatBoost.xlsx
    model_name="TabNet",
    n_quantiles=10
)

✅ Calibration curve & metrics saved to Excel:
   drive/MyDrive/Regression/noise_uncertainty_analysis/probabilistic_distribution_card/TabNet.xlsx


,Predicted_Quantile,Observed_Frequency,Absolute_Error
0,0.0,0.000000,0.000000e+00
1,0.1,0.150000,5.000000e-02
2,0.2,0.216667,1.666667e-02
3,0.3,0.300000,5.551115e-17
4,0.4,0.383333,1.666667e-02
5,0.5,0.483333,1.666667e-02
6,0.6,0.550000,5.000000e-02
7,0.7,0.600000,1.000000e-01
8,0.8,0.750000,5.000000e-02
9,0.9,0.900000,0.000000e+00


# **Probabilistic distributions using PGBM**

In [60]:
# Assuming best_params is obtained correctly
best_params = get_best_model_params(best_scores_autosampler, 'PGBM')

# Remove any conflicting parameters (safe-guard)
for key in ['Dist', 'Score']:
    best_params.pop(key, None)

# -------------------------
# Train PGBM as BASE model
# -------------------------
pgbm_model = PGBM()

def mseloss_objective(yhat, y, sample_weight=None):
    if not torch.is_tensor(yhat):
        yhat = torch.from_numpy(np.array(yhat)).float()
    if not torch.is_tensor(y):
        y = torch.from_numpy(np.array(y)).float()
    gradient = yhat - y
    hessian = torch.ones_like(yhat)
    return gradient, hessian

def rmseloss_metric(yhat, y, sample_weight=None):
    if not torch.is_tensor(yhat):
        yhat = torch.from_numpy(np.array(yhat)).float()
    if not torch.is_tensor(y):
        y = torch.from_numpy(np.array(y)).float()
    return torch.sqrt(torch.mean((yhat - y) ** 2))

X_train_np = np.array(X_train)
y_train_np = np.array(y_train)
X_test_np = np.array(X_test)
y_test_np = np.array(y_test)

pgbm_model.train(
    (X_train_np, y_train_np),
    objective=mseloss_objective,
    metric=rmseloss_metric,
    params=best_params
)

# -------------------------
# 🔧 NUMPY-SAFE WRAPPER
# -------------------------
class PGBMWrapper:
    def __init__(self, model):
        self.model = model

    def predict(self, X):
        y_pred = self.model.predict(X)
        if torch.is_tensor(y_pred):
            y_pred = y_pred.detach().cpu().numpy()
        return y_pred

pgbm_wrapped = PGBMWrapper(pgbm_model)

# -------------------------
# CARD wrapper on top
# -------------------------
card_model = CARDRegressor(  # FIX C1: official CARD settings (T, epochs, n_z_samples)
    base_model=pgbm_wrapped,
    hidden_dim=128,
    T=1000,
    epochs=5000,
    n_samples=1000
)

# Fit CARD (NOW SAFE)
card_model.fit(X_train_np, y_train_np)

# CARD predictions
mean_pred, std_pred = card_model.predict(X_test_np)

# Create prediction DataFrame
predictions_PGBM_df = pd.DataFrame({
    "Mean": mean_pred,
    "StdDev": std_pred,
    "Actual": y_test_np.ravel()
})

# Calculate and print MSE
mse = mean_squared_error(y_test_np, mean_pred)
print(f"Mean Squared Error: {mse:.2f}")

# Preview results
print(predictions_PGBM_df.head())

Training on CPU


Estimator 0/100, Train metric: 13.5823
Estimator 1/100, Train metric: 12.9859
Estimator 2/100, Train metric: 12.4356
Estimator 3/100, Train metric: 11.9029
Estimator 4/100, Train metric: 11.4019
Estimator 5/100, Train metric: 10.9165
Estimator 6/100, Train metric: 10.4506
Estimator 7/100, Train metric: 10.0087
Estimator 8/100, Train metric: 9.5896
Estimator 9/100, Train metric: 9.1922
Estimator 10/100, Train metric: 8.8209
Estimator 11/100, Train metric: 8.4728
Estimator 12/100, Train metric: 8.1313


Estimator 13/100, Train metric: 7.8142
Estimator 14/100, Train metric: 7.5028
Estimator 15/100, Train metric: 7.2011
Estimator 16/100, Train metric: 6.9205
Estimator 17/100, Train metric: 6.6474
Estimator 18/100, Train metric: 6.3886
Estimator 19/100, Train metric: 6.1544
Estimator 20/100, Train metric: 5.9238
Estimator 21/100, Train metric: 5.7004
Estimator 22/100, Train metric: 5.5008
Estimator 23/100, Train metric: 5.2968
Estimator 24/100, Train metric: 5.1073
Estimator 25/100, Train metric: 4.9373
Estimator 26/100, Train metric: 4.7665
Estimator 27/100, Train metric: 4.6124


Estimator 28/100, Train metric: 4.4596
Estimator 29/100, Train metric: 4.3087
Estimator 30/100, Train metric: 4.1734
Estimator 31/100, Train metric: 4.0334
Estimator 32/100, Train metric: 3.9122
Estimator 33/100, Train metric: 3.7774
Estimator 34/100, Train metric: 3.6682
Estimator 35/100, Train metric: 3.5549
Estimator 36/100, Train metric: 3.4547
Estimator 37/100, Train metric: 3.3491
Estimator 38/100, Train metric: 3.2461
Estimator 39/100, Train metric: 3.1490
Estimator 40/100, Train metric: 3.0611


Estimator 41/100, Train metric: 2.9699
Estimator 42/100, Train metric: 2.8949
Estimator 43/100, Train metric: 2.8201
Estimator 44/100, Train metric: 2.7370
Estimator 45/100, Train metric: 2.6646
Estimator 46/100, Train metric: 2.6019
Estimator 47/100, Train metric: 2.5310
Estimator 48/100, Train metric: 2.4609
Estimator 49/100, Train metric: 2.3927
Estimator 50/100, Train metric: 2.3280


Estimator 51/100, Train metric: 2.2702
Estimator 52/100, Train metric: 2.2238
Estimator 53/100, Train metric: 2.1783
Estimator 54/100, Train metric: 2.1239
Estimator 55/100, Train metric: 2.0783
Estimator 56/100, Train metric: 2.0355
Estimator 57/100, Train metric: 1.9918
Estimator 58/100, Train metric: 1.9570
Estimator 59/100, Train metric: 1.9246


... (3 outputs left out)

Estimator 83/100, Train metric: 1.3184
Estimator 84/100, Train metric: 1.3006
Estimator 85/100, Train metric: 1.2895
Estimator 86/100, Train metric: 1.2736
Estimator 87/100, Train metric: 1.2586
Estimator 88/100, Train metric: 1.2463
Estimator 89/100, Train metric: 1.2324


Estimator 90/100, Train metric: 1.2171
Estimator 91/100, Train metric: 1.1993
Estimator 92/100, Train metric: 1.1860
Estimator 93/100, Train metric: 1.1720
Estimator 94/100, Train metric: 1.1546
Estimator 95/100, Train metric: 1.1393
Estimator 96/100, Train metric: 1.1250


Estimator 97/100, Train metric: 1.1154
Estimator 98/100, Train metric: 1.0994
Estimator 99/100, Train metric: 1.0854


Mean Squared Error: 20.07
        Mean    StdDev  Actual
0  72.711034  1.034373   70.07
1  76.097982  0.939566   72.90
2  61.803843  1.093608   68.66
3  64.013377  1.122024   50.24
4  38.539207  1.337656   40.17


In [61]:
generate_and_save_plots(predictions_PGBM_df, y_test, "./drive/MyDrive/Regression/noise_uncertainty_analysis/probabilistic_distribution_card/PGBM.xlsx")

Excel file saved at ./drive/MyDrive/Regression/noise_uncertainty_analysis/probabilistic_distribution_card/PGBM.xlsx


In [62]:
generate_and_save_crps_decomposition(
    predictions_PGBM_df,
    y_test,
    "./drive/MyDrive/Regression/noise_uncertainty_analysis/probabilistic_distribution_card/PGBM.xlsx",
    model_name="PGBM",
    n_bins=N_BINS
)

✅ CRPS decomposition saved to:
drive/MyDrive/Regression/noise_uncertainty_analysis/probabilistic_distribution_card/PGBM.xlsx
   Sheets: CRPS_Summary_PGBM, CRPS_Bins_PGBM


,Model,Total_CRPS,Reliability,Resolution,Uncertainty,Num_Bins
0,PGBM,2.513336,0.396682,5.761718,7.879017,6


In [63]:
generate_and_save_pit_results(
    predictions_PGBM_df,
    y_test,
    "./drive/MyDrive/Regression/noise_uncertainty_analysis/probabilistic_distribution_card/PGBM.xlsx",
    model_name="PGBM",
    n_bins=N_BINS
)

✅ PIT values, histogram, summary, and EMBEDDED plot saved to:
   drive/MyDrive/Regression/noise_uncertainty_analysis/probabilistic_distribution_card/PGBM.xlsx


,PIT_Value
0,5.335844e-03
1,3.324222e-04
2,1.000000e+00
3,6.133471e-35
4,8.886048e-01
5,9.999995e-01
6,4.324741e-01
7,4.207203e-01
8,3.805393e-05
9,5.812972e-03


In [64]:
generate_and_save_calibration_curve(
    predictions_PGBM_df,
    y_test,
    "./drive/MyDrive/Regression/noise_uncertainty_analysis/probabilistic_distribution_card/PGBM.xlsx",
    model_name="PGBM",
    n_quantiles=10
)

✅ Calibration curve & metrics saved to Excel:
   drive/MyDrive/Regression/noise_uncertainty_analysis/probabilistic_distribution_card/PGBM.xlsx


,Predicted_Quantile,Observed_Frequency,Absolute_Error
0,0.0,0.000000,0.000000
1,0.1,0.366667,0.266667
2,0.2,0.416667,0.216667
3,0.3,0.466667,0.166667
4,0.4,0.533333,0.133333
5,0.5,0.600000,0.100000
6,0.6,0.666667,0.066667
7,0.7,0.666667,0.033333
8,0.8,0.716667,0.083333
9,0.9,0.800000,0.100000


# **Matrix Evaulation**

In [65]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

# Create a Pandas Excel writer using XlsxWriter as the engine.
folder_path = "/content/drive/MyDrive/Regression/noise_uncertainty_analysis"
excel_path = os.path.join(folder_path, "/content/drive/MyDrive/Regression/noise_uncertainty_analysis/probabilistic_distribution_card/Matrix Evaluation.xlsx")
with pd.ExcelWriter(_ensure_parent_dir(excel_path), engine='xlsxwriter') as writer:
    plot_metrics_and_save_to_excel(writer, predictions_NGB_df, y_test, 'NGBoost')
    plot_metrics_and_save_to_excel(writer, predictions_PGBM_df, y_test, 'PGBM')
    plot_metrics_and_save_to_excel(writer, predictions_LGBM_df, y_test, 'LGBM')
    plot_metrics_and_save_to_excel(writer, predictions_XGB_df, y_test, 'XGBoost')
    plot_metrics_and_save_to_excel(writer, predictions_TabNet_df, y_test, "TabNet")
    plot_metrics_and_save_to_excel(writer, predictions_HGBR_df, y_test, "HGBM")
    plot_metrics_and_save_to_excel(writer, predictions_CatBoost_df, y_test, "CatBoost")
    plot_metrics_and_save_to_excel(writer, predictions_GPBoost_df, y_test, "GPBoost")
    plot_metrics_and_save_to_excel(writer, predictions_GBR_df, y_test, "GradientBoosting")